# §4 Prior Exploitation — Instruction Effect: Blind → Inst-Blind

Dedicated comparison of model behaviour under two no-image conditions:
- **blind** — question only, no instruction, no image
- **inst_blind** — same, plus: *"No image is provided. Answer from language priors only."*

This notebook captures the **gating effect**: a single instruction sentence activates committed hallucination.

Sections:
1. Accuracy Δ: blind vs inst_blind (per model, per variant)
2. Answer change rate (what fraction of answers flip?)
3. Soft abstention collapse (prior unlocked by instruction)
4. 4-way transition matrix (abstained/committed × blind/inst)
5. Confidence shift: LP distribution blind → inst_blind
6. Quadrant expansion: does inst_blind grow the LH (prior exploitation) zone vs blind?

In [ ]:
import json, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

BASE = Path('/home/david/Desktop/yuna/HPA')
sys.path.insert(0, str(BASE / 'analysis'))

from utils.vqa import VQAAnswerMapper, vqa_accuracy
from utils.load_session import load_jsonl, clean_answer as _clean_answer, load_human_data
from utils.constants import VARIANT_ORDER, VARIANT_LABELS, VARIANT_COLORS

mapper = VQAAnswerMapper()

CT_TO_VAR = {'question': 'C', 'weaker_object': 'B', 'pronominalized': 'A'}

# All VLMs with both blind + inst_blind
ALL_MODELS = {
    'Qwen3-VL-8B':   BASE / 'evaluation/logits/pretrained/Qwen3-VL-8B-Instruct',
    'LLaVA-1.5-7B':  BASE / 'evaluation/logits/pretrained/llava-1.5-7b-hf',
    'LLaVA-Mistral': BASE / 'evaluation/logits/pretrained/llava-v1.6-mistral-7b-hf',
    'LLaVA-Vicuna':  BASE / 'evaluation/logits/pretrained/llava-v1.6-vicuna-7b-hf',
    'InternVL-1B':   BASE / 'evaluation/logits/pretrained/InternVL3_5-1B',
    'InternVL-2B':   BASE / 'evaluation/logits/pretrained/InternVL3_5-2B',
    'InternVL-8B':   BASE / 'evaluation/logits/pretrained/InternVL3_5-8B',
}

SOFT_WORDS = {
    'nothing', 'none', 'nowhere', 'unanswerable', 'unknown', 'unclear',
    'cannot', "can't", 'no image', 'blank', 'n/a', 'not visible', 'not shown',
    'indeterminate', 'unidentifiable', 'no sign', "can't tell", 'cannot tell',
    'not clear',
}

def is_abstain(ans):
    a = ans.lower().strip()
    return any(w in a for w in SOFT_WORDS)

print('Models:', list(ALL_MODELS.keys()))

In [ ]:
# ── Load blind + inst_blind answers for all models ────────────────────────────
# per_model[label] = {'blind': {qid: {var: ans}}, 'inst_blind': {qid: {var: ans}}}
per_model = {}

for label, mdir in ALL_MODELS.items():
    per_model[label] = {}
    for cond, fname in [('blind', 'vqa_1k_control_blind'),
                        ('inst_blind', 'vqa_1k_control_inst_blind')]:
        d = {}
        for ex in load_jsonl(mdir / f'{fname}.jsonl'):
            qid = int(ex['question_id'])
            ga  = ex.get('generated_answers', {})
            d[qid] = {var: _clean_answer(ga.get(ct, ''))
                      for ct, var in CT_TO_VAR.items()}
        per_model[label][cond] = d
    b_n = len(per_model[label]['blind'])
    i_n = len(per_model[label]['inst_blind'])
    print(f'  {label}: blind={b_n}  inst_blind={i_n}')

all_qids = set.intersection(*[
    set(per_model[l]['blind'].keys()) & set(per_model[l]['inst_blind'].keys())
    for l in ALL_MODELS
])
print(f'\nCommon questions across all models: {len(all_qids)}')

## 1. Accuracy Δ: Blind vs Inst-Blind (per model, per variant)

In [ ]:
# Compute per-model accuracy under both conditions
rows = []
for label in ALL_MODELS:
    for cond in ['blind', 'inst_blind']:
        for var in VARIANT_ORDER:
            accs = []
            for qid, vd in per_model[label][cond].items():
                ans = vd.get(var, '')
                gt  = mapper.get_answers(qid)
                if ans and gt:
                    accs.append(vqa_accuracy(ans, gt))
            rows.append({'model': label, 'condition': cond, 'variant': var,
                         'accuracy': np.mean(accs) if accs else np.nan})

acc_df = pd.DataFrame(rows)
pivot  = acc_df.pivot_table(index='model', columns=['condition', 'variant'], values='accuracy')

# Delta table
delta_rows = []
for label in ALL_MODELS:
    for var in VARIANT_ORDER:
        b = acc_df[(acc_df.model == label) & (acc_df.condition == 'blind')    & (acc_df.variant == var)]['accuracy'].values
        i = acc_df[(acc_df.model == label) & (acc_df.condition == 'inst_blind') & (acc_df.variant == var)]['accuracy'].values
        delta_rows.append({'model': label, 'variant': var,
                           'blind': b[0] if len(b) else np.nan,
                           'inst_blind': i[0] if len(i) else np.nan,
                           'delta': (i[0] - b[0]) if (len(b) and len(i)) else np.nan})

delta_df = pd.DataFrame(delta_rows)
print('Accuracy: blind → inst_blind (delta = inst - blind)')
print(delta_df.pivot_table(index='model', columns='variant',
                            values=['blind', 'inst_blind', 'delta']).round(3))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
models = list(ALL_MODELS.keys())
x = np.arange(len(models))
w = 0.35

for ax, var in zip(axes, VARIANT_ORDER):
    sub = delta_df[delta_df.variant == var].set_index('model').reindex(models)
    ax.bar(x - w/2, sub['blind'],      w, label='blind',      color='#e74c3c', alpha=0.8)
    ax.bar(x + w/2, sub['inst_blind'], w, label='inst_blind', color='#3498db', alpha=0.8)
    for xi, (b_val, i_val) in enumerate(zip(sub['blind'], sub['inst_blind'])):
        delta = i_val - b_val
        ax.annotate(f'{delta:+.2f}', (xi, max(b_val, i_val) + 0.01),
                    ha='center', fontsize=7,
                    color='#2980b9' if delta > 0 else '#c0392b')
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('LLaVA-','') for m in models],
                       rotation=25, ha='right', fontsize=8)
    ax.set_title(f'Variant {var} — {VARIANT_LABELS[var]}')
    ax.set_ylim(0, 0.75)
    if ax == axes[0]:
        ax.set_ylabel('VQA Accuracy')
        ax.legend(fontsize=8)

plt.suptitle('Accuracy: Blind vs Inst-Blind (Δ annotated)', y=1.02)
plt.tight_layout()
plt.show()

## 2. Answer Change Rate

In [ ]:
change_rows = []
for label in ALL_MODELS:
    common = set(per_model[label]['blind'].keys()) & set(per_model[label]['inst_blind'].keys())
    for var in VARIANT_ORDER:
        changed = sum(
            per_model[label]['blind'][qid].get(var, '').lower().strip() !=
            per_model[label]['inst_blind'][qid].get(var, '').lower().strip()
            for qid in common
        )
        change_rows.append({'model': label, 'variant': var,
                            'change_rate': changed / len(common) if common else np.nan,
                            'n': len(common)})

chg_df = pd.DataFrame(change_rows)
print('Answer change rate (fraction of questions where answer differs blind → inst_blind):')
print(chg_df.pivot_table(index='model', columns='variant', values='change_rate').round(3))

fig, ax = plt.subplots(figsize=(10, 4))
models = list(ALL_MODELS.keys())
x = np.arange(len(models))
w = 0.25
offsets = [-w, 0, w]
for var, off in zip(VARIANT_ORDER, offsets):
    vals = [chg_df[(chg_df.model==m) & (chg_df.variant==var)]['change_rate'].values[0]
            for m in models]
    ax.bar(x + off, vals, w, label=VARIANT_LABELS[var],
           color=VARIANT_COLORS[var], alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Fraction of questions with changed answer')
ax.set_title('Answer Change Rate: Blind → Inst-Blind')
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', ls='--', lw=0.8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Soft Abstention Collapse

Of questions where the model soft-abstained under **blind** (output 'nothing', 'unknown', etc.),
what fraction become committed answers under **inst_blind**?

High collapse rate = the model *had* the prior but was suppressing it without explicit permission.

In [ ]:
collapse_rows = []
for label in ALL_MODELS:
    common = set(per_model[label]['blind'].keys()) & set(per_model[label]['inst_blind'].keys())
    for var in VARIANT_ORDER:
        soft_blind = [qid for qid in common
                      if is_abstain(per_model[label]['blind'][qid].get(var, ''))]
        collapsed  = [qid for qid in soft_blind
                      if not is_abstain(per_model[label]['inst_blind'][qid].get(var, ''))]
        still_abst = [qid for qid in common
                      if is_abstain(per_model[label]['inst_blind'][qid].get(var, ''))]
        collapse_rows.append({
            'model': label, 'variant': var,
            'soft_blind_pct': len(soft_blind) / len(common) if common else np.nan,
            'collapse_pct':   len(collapsed)  / len(soft_blind) if soft_blind else np.nan,
            'soft_inst_pct':  len(still_abst) / len(common) if common else np.nan,
            'n_soft_blind': len(soft_blind),
            'n_collapsed':  len(collapsed),
        })

col_df = pd.DataFrame(collapse_rows)

print('Soft abstention rates and collapse (variant C):')
c_df = col_df[col_df.variant == 'C'][['model','soft_blind_pct','collapse_pct','soft_inst_pct','n_soft_blind','n_collapsed']]
print(c_df.set_index('model').round(3))

# Plot: soft abstention rate blind vs inst + collapse rate
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
models = list(ALL_MODELS.keys())
x = np.arange(len(models))
w = 0.3

ax = axes[0]
for var, off in zip(VARIANT_ORDER, [-w, 0, w]):
    vals = [col_df[(col_df.model==m) & (col_df.variant==var)]['soft_blind_pct'].values[0]
            for m in models]
    ax.bar(x + off, vals, w, label=VARIANT_LABELS[var],
           color=VARIANT_COLORS[var], alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Fraction of questions')
ax.set_title('Soft Abstention Rate — Blind condition')
ax.legend(fontsize=8)
ax.set_ylim(0, 0.35)

ax = axes[1]
for var, off in zip(VARIANT_ORDER, [-w, 0, w]):
    vals = [col_df[(col_df.model==m) & (col_df.variant==var)]['collapse_pct'].values[0]
            for m in models]
    ax.bar(x + off, vals, w, label=VARIANT_LABELS[var],
           color=VARIANT_COLORS[var], alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Collapse rate (of blind soft-abstentions)')
ax.set_title('Soft Abstention Collapse Rate: Blind → Inst-Blind')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', ls='--', lw=0.8)

plt.tight_layout()
plt.show()

## 4. 4-Way Transition Matrix

| blind \ inst_blind | abstained | committed |
|---|---|---|
| **abstained** | stable abstain | **collapse** (prior unlocked) |
| **committed** | late abstain (rare) | stable hallucination |

In [ ]:
# Variant C only — clearest signal
fig, axes = plt.subplots(1, len(ALL_MODELS), figsize=(3 * len(ALL_MODELS), 3.5))

for ax, label in zip(axes, ALL_MODELS):
    common = set(per_model[label]['blind'].keys()) & set(per_model[label]['inst_blind'].keys())
    mat = np.zeros((2, 2), dtype=int)
    # rows: blind (0=abstain, 1=committed)
    # cols: inst  (0=abstain, 1=committed)
    for qid in common:
        b = per_model[label]['blind'][qid].get('C', '')
        i = per_model[label]['inst_blind'][qid].get('C', '')
        r = 0 if is_abstain(b) else 1
        c = 0 if is_abstain(i) else 1
        mat[r, c] += 1
    n = len(common)
    mat_pct = mat / n

    im = ax.imshow(mat_pct, cmap='Blues', vmin=0, vmax=1)
    labels_rc = [['Stable\nabstain', 'COLLAPSE\n(prior unlocked)'],
                 ['Late abstain\n(rare)', 'Stable\nhallucination']]
    for r in range(2):
        for c in range(2):
            v = mat_pct[r, c]
            ax.text(c, r, f'{labels_rc[r][c]}\n{mat[r,c]} ({100*v:.0f}%)',
                    ha='center', va='center', fontsize=7,
                    color='white' if v > 0.5 else 'black')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['abstained', 'committed'], fontsize=8)
    ax.set_yticklabels(['abstained\n(blind)', 'committed\n(blind)'], fontsize=8)
    ax.set_title(label.replace('LLaVA-', '').replace('InternVL-', 'IVL-'),
                 fontsize=9, fontweight='bold')

plt.suptitle('Blind → Inst-Blind Transition Matrix (Variant C)', y=1.02)
plt.tight_layout()
plt.show()

print('\nKey insight: COLLAPSE cell = prior exploitation gated by instruction.')
print('High collapse rate = model has the prior, only needs permission to express it.')

## 5. Confidence Shift: Blind → Inst-Blind

In [ ]:
# Load logprob confidence for models that have it (checking generated_logits)
# Mean token logprob (sum/len) per answer as confidence proxy
conf_rows = []
for label, mdir in ALL_MODELS.items():
    for cond, fname in [('blind', 'vqa_1k_control_blind'),
                        ('inst_blind', 'vqa_1k_control_inst_blind')]:
        n_with_logits = 0
        for ex in load_jsonl(mdir / f'{fname}.jsonl'):
            gl = ex.get('generated_logits', {})
            if not gl: continue
            qid = int(ex['question_id'])
            for ct, var in CT_TO_VAR.items():
                ct_logits = gl.get(ct, {}).get('content', [])
                if not ct_logits: continue
                lps = [t['logprob'] for t in ct_logits]
                conf_rows.append({
                    'model': label, 'condition': cond, 'variant': var,
                    'question_id': qid,
                    'mean_lp': float(np.mean(lps)),
                })
                n_with_logits += 1
        if n_with_logits > 0:
            print(f'  {label}/{cond}: {n_with_logits} logprob records')

if conf_rows:
    conf_df = pd.DataFrame(conf_rows)

    # Mean LP shift per model (variant C)
    lp_pivot = (conf_df[conf_df.variant == 'C']
                .groupby(['model', 'condition'])['mean_lp'].mean()
                .unstack())
    if 'blind' in lp_pivot and 'inst_blind' in lp_pivot:
        lp_pivot['delta'] = lp_pivot['inst_blind'] - lp_pivot['blind']
        print('\nMean log-prob (variant C): blind → inst_blind')
        print(lp_pivot.round(4))

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        # Left: LP distributions
        ax = axes[0]
        for cond, color, alpha in [('blind', '#e74c3c', 0.5), ('inst_blind', '#3498db', 0.5)]:
            vals = conf_df[(conf_df.condition == cond) & (conf_df.variant == 'C')]['mean_lp']
            ax.hist(vals, bins=40, color=color, alpha=alpha,
                    label=f'{cond} (μ={vals.mean():.3f})', density=True)
        ax.set_xlabel('Mean log-prob per answer')
        ax.set_ylabel('Density')
        ax.set_title('Confidence (LP) Distribution: Blind vs Inst-Blind (variant C, all models)')
        ax.legend(fontsize=9)

        # Right: per-model delta
        ax = axes[1]
        models_with_lp = [m for m in ALL_MODELS if m in lp_pivot.index]
        deltas = [lp_pivot.loc[m, 'delta'] for m in models_with_lp]
        colors = ['#2980b9' if d > 0 else '#c0392b' for d in deltas]
        ax.barh(models_with_lp, deltas, color=colors, alpha=0.85, edgecolor='white')
        ax.axvline(0, color='black', lw=0.8)
        ax.set_xlabel('Δ Mean LP (inst_blind − blind)')
        ax.set_title('Confidence Shift per Model (variant C)\nPositive = more confident under inst_blind')
        plt.tight_layout()
        plt.show()
else:
    print('No logprob data found for these models — confidence analysis skipped.')
    print('(Logprobs available for: Vicuna-7b, Vicuna-13b, Qwen3-VL-2B, Qwen3-VL-4B)')

## 6. Quadrant Expansion: Does Inst-Blind Grow the LH Zone?

The LH quadrant (model correct, human wrong) is the **prior exploitation zone**.
Does the instruction expand it — i.e. does inst_blind move more questions into LH vs blind?

Human data: inst_blind condition (participants told no image).
Model: compared under blind vs inst_blind.
THR = 0.5 accuracy threshold for quadrant assignment.

In [ ]:
# Load human data
participants, common_qids, h_df, mapper2 = load_human_data(BASE, min_answers=348, verbose=False)
h_acc = h_df.groupby(['question_id', 'variant'])['accuracy'].mean().unstack()['C']
h_qids = set(h_acc.index)

THR = 0.5

quad_labels = {
    'both_right':  'Both right',
    'human_right': 'Human right\nModel wrong',
    'model_right': 'Model right\nHuman wrong\n(LH zone)',
    'both_wrong':  'Both wrong',
}
quad_colors = {
    'both_right':  '#27ae60',
    'human_right': '#3498db',
    'model_right': '#e74c3c',
    'both_wrong':  '#95a5a6',
}
quad_order = ['both_right', 'human_right', 'model_right', 'both_wrong']

def get_quadrant(h, m):
    if pd.isna(h) or pd.isna(m): return None
    if h >= THR and m >= THR: return 'both_right'
    if h >= THR and m <  THR: return 'human_right'
    if h <  THR and m >= THR: return 'model_right'
    return 'both_wrong'

# Compute mean model accuracy per question for blind and inst_blind
for_cond = {}
for cond in ['blind', 'inst_blind']:
    qid_accs = defaultdict(list)
    for label in ALL_MODELS:
        for qid, vd in per_model[label][cond].items():
            if qid not in h_qids: continue
            ans = vd.get('C', '')
            gt  = mapper.get_answers(qid)
            if ans and gt:
                qid_accs[qid].append(vqa_accuracy(ans, gt))
    for_cond[cond] = {qid: np.mean(v) for qid, v in qid_accs.items() if v}

# Build comparison DataFrame
quad_rows = []
for qid in sorted(h_qids & set(for_cond['blind']) & set(for_cond['inst_blind'])):
    h = h_acc.get(qid, np.nan)
    quad_rows.append({
        'question_id': qid,
        'human_acc':   h,
        'model_blind': for_cond['blind'][qid],
        'model_inst':  for_cond['inst_blind'][qid],
        'quad_blind':  get_quadrant(h, for_cond['blind'][qid]),
        'quad_inst':   get_quadrant(h, for_cond['inst_blind'][qid]),
    })
quad_df = pd.DataFrame(quad_rows)

# Summary
n = len(quad_df)
print(f'Questions with human + model data: {n}')
print(f'\n{"Quadrant":30}  {"Blind":>8}  {"Inst-Blind":>10}  {"Δ":>6}')
print('-' * 60)
for q in quad_order:
    nb = (quad_df.quad_blind == q).sum()
    ni = (quad_df.quad_inst  == q).sum()
    label = quad_labels[q].replace('\n', ' ')
    marker = ' ← KEY' if q == 'model_right' else ''
    print(f'{label:30}  {nb:5d} ({100*nb/n:.0f}%)  {ni:5d} ({100*ni/n:.0f}%)  {ni-nb:+4d}{marker}')

# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cond, col in [(axes[0], 'blind', 'quad_blind'),
                      (axes[1], 'inst_blind', 'quad_inst')]:
    mat = np.array([[( (quad_df[col]=='both_right').sum()),
                     ( (quad_df[col]=='human_right').sum())],
                    [( (quad_df[col]=='model_right').sum()),
                     ( (quad_df[col]=='both_wrong').sum())]])
    ax.imshow(mat, cmap='Blues', vmin=0, vmax=n)
    for r in range(2):
        for c in range(2):
            v = mat[r, c]
            ax.text(c, r, f'{v}\n({100*v/n:.0f}%)', ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='white' if v > n * 0.3 else 'black')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Model\ncorrect', 'Model\nwrong'])
    ax.set_yticklabels(['Human\ncorrect', 'Human\nwrong'])
    ax.set_title(f'Model condition: {cond}\n(human always inst_blind)', fontsize=10)

plt.suptitle('Quadrant Distribution: Blind vs Inst-Blind (all-model mean, variant C)', y=1.02)
plt.tight_layout()
plt.show()

# Transition between quadrants
print('\nQuestion transitions blind → inst_blind:')
trans = quad_df.groupby(['quad_blind', 'quad_inst']).size().unstack(fill_value=0)
print(trans)